In [1]:
from pathlib import Path

import airbench
import torch
import torch.nn.functional as F
from airbench.utils import CIFAR_MEAN, CIFAR_STD
from tqdm.autonotebook import tqdm

from model import CifarVaeModel

/tmp/ipykernel_4110353/2298137818.py:7: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


### Define and load data

In [ ]:
DATA_DIR = Path.home() / ".cache" / "airbench-cifar10"
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_loader = airbench.CifarLoader(
    str(DATA_DIR),
    train=True,
    batch_size=1024,
    aug=dict(flip=True, translate=4, cutout=16),
)
test_loader = airbench.CifarLoader(
    str(DATA_DIR),
    train=False,
    batch_size=1000,
)

# VAE serves as a tokenizer for downstream generative modeling — train on all 60k.
train_loader.images = torch.cat([train_loader.images, test_loader.images])
train_loader.labels = torch.cat([train_loader.labels, test_loader.labels])
print(f"train: {len(train_loader.images)} images, {len(train_loader)} batches of {train_loader.batch_size}")

train: 60000 images, 117 batches of 512


### Define model

In [ ]:
device = "cuda"
model = CifarVaeModel(base_c=64, ch_mults=(1, 2, 4), z_c=4, n_res=2).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

params: 7.06M


## VAE training loop

In [4]:
beta = 1.0
epochs = 5

model.train()
for epoch in range(epochs):
    pbar = tqdm(train_loader, desc=f"epoch {epoch}", leave=True)
    for images, _ in pbar:
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            x_hat, mu, logvar = model(images)
            recon = F.mse_loss(x_hat, images, reduction="none").sum(dim=(1, 2, 3)).mean()
            kl = 0.5 * (mu.pow(2) + logvar.exp() - 1 - logvar).sum(dim=(1, 2, 3)).mean()
            loss = recon + beta * kl
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        pbar.set_postfix(recon=f"{recon.item():.2f}", kl=f"{kl.item():.2f}")

epoch 0:   0%|          | 0/117 [00:00<?, ?it/s]

epoch 1:   0%|          | 0/117 [00:00<?, ?it/s]

epoch 2:   0%|          | 0/117 [00:00<?, ?it/s]

epoch 3:   0%|          | 0/117 [00:00<?, ?it/s]

epoch 4:   0%|          | 0/117 [00:00<?, ?it/s]

### Post-training

In [5]:
# After training, save to /home/nlyu/Code/diffusive-latent-generation/experiments/cifar-smoke VAE / encoder.pkl, decoder.pkl
# Also save the image artifacts (train + val)